<a href="https://colab.research.google.com/github/manya-ghorpade/CollegeWorks/blob/main/LLM/LLM_V.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip uninstall torchvision torchaudio -y

Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu
Found existing installation: torchaudio 2.10.0+cpu
Uninstalling torchaudio-2.10.0+cpu:
  Successfully uninstalled torchaudio-2.10.0+cpu


In [ ]:
!pip install torchvision==0.17.0 torchaudio==2.2.0
# !pip install -q transformers==4.39.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 26.4 MB/s eta 0:00:00


In [ ]:
!pip install transformers==4.41.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 31.8 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.39.3
    Uninstalling transformers-4.39.3:
      Successfully uninstalled transformers-4.39.3


In [ ]:
!pip install -q peft==0.10.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 4.0 MB/s eta 0:00:00


In [ ]:
!pip install -q accelerate==0.28.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 6.2 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets==2.18.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.


In [ ]:
import transformers
import torch
import peft

print(transformers.__version__)
print(torch.__version__)
print(peft.__version__)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

4.39.3
2.2.0+cu121
0.10.0


In [ ]:
# ============================================================
# PEFT LoRA Fine-Tuning on Google Colab (Stable Version)
# ============================================================

# ------------------------------------------------------------
# Import Required Libraries
# ------------------------------------------------------------

import torch
# PyTorch – core deep learning framework used for model training

from datasets import load_dataset
# Hugging Face Datasets – used to load benchmark datasets easily

from transformers import (
    AutoTokenizer,                 # Automatically loads correct tokenizer
    AutoModelForCausalLM,          # Loads pretrained model for text generation
    TrainingArguments,             # Defines training hyperparameters
    Trainer,                       # High-level training API
    DataCollatorForLanguageModeling # Prepares batches for causal LM
)

from peft import LoraConfig, get_peft_model, TaskType
# PEFT library – enables parameter-efficient fine-tuning (LoRA)

# ------------------------------------------------------------
# Check Device (CPU or GPU)
# ------------------------------------------------------------

# Detect whether GPU is available in Colab
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# ------------------------------------------------------------
# 1. Load Dataset
# ------------------------------------------------------------

# Load WikiText-2 dataset (raw version)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# Use only 2000 samples for faster training demonstration
dataset = dataset["train"].select(range(2000))

# ------------------------------------------------------------
# 2. Load Pretrained Model + Tokenizer
# ------------------------------------------------------------

# Name of pretrained model
model_name = "distilgpt2"

# Load tokenizer corresponding to the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT2 models do not have a padding token by default
# So we set pad_token equal to end-of-sequence token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load pretrained GPT2 model for causal language modeling
model = AutoModelForCausalLM.from_pretrained(model_name)

# Resize token embeddings (important after modifying tokenizer)
model.resize_token_embeddings(len(tokenizer))

# Move model to GPU (if available)
model.to(device)

# ------------------------------------------------------------
# 3. Tokenization Function
# ------------------------------------------------------------

# Convert raw text into token IDs
def tokenize_function(examples):
    return tokenizer(
        examples["text"],       # Input text column
        truncation=True,        # Cut sequences longer than max_length
        padding="max_length",   # Pad shorter sequences
        max_length=128          # Fixed sequence length
    )

# Apply tokenization to entire dataset
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,              # Process multiple samples at once
    remove_columns=["text"]    # Remove original text column
)

# ------------------------------------------------------------
# 4. Apply LoRA (Parameter-Efficient Fine-Tuning)
# ------------------------------------------------------------

# Define LoRA configuration
lora_config = LoraConfig(
    r=8,                        # Rank of low-rank matrices
    lora_alpha=16,              # Scaling factor
    target_modules=["c_attn"],  # Apply LoRA to attention layers
    lora_dropout=0.1,           # Dropout for LoRA layers
    bias="none",                # Do not train bias
    task_type=TaskType.CAUSAL_LM # Task type = text generation
)

# Inject LoRA adapters into model
model = get_peft_model(model, lora_config)

# Print how many parameters are trainable
# You will see only ~1% parameters are trained
model.print_trainable_parameters()

# ------------------------------------------------------------
# 5. Data Collator
# ------------------------------------------------------------

# Automatically prepares batches for causal language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # False because GPT2 uses causal LM (not masked LM)
)

# ------------------------------------------------------------
# 6. Define Training Hyperparameters
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./lora-gpt2",      # Where model checkpoints are saved
    per_device_train_batch_size=8,# Batch size per GPU
    num_train_epochs=1,           # Number of training epochs
    logging_steps=50,             # Log every 50 steps
    save_steps=500,               # Save checkpoint every 500 steps
    learning_rate=2e-4,           # Learning rate for optimizer
    fp16=torch.cuda.is_available(), # Mixed precision for faster training
    report_to="none"              # Disable WandB/other logging
)

# ------------------------------------------------------------
# 7. Initialize Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,                      # Model with LoRA adapters
    args=training_args,               # Training settings
    train_dataset=tokenized_dataset,  # Tokenized dataset
    data_collator=data_collator       # Batch processor
)

# ------------------------------------------------------------
# 8. Start Fine-Tuning
# ------------------------------------------------------------

# Begin training process
trainer.train()

# ------------------------------------------------------------
# 9. Save Fine-Tuned LoRA Model
# ------------------------------------------------------------

# Save only LoRA adapter weights (very small size)
model.save_pretrained("./lora-gpt2")

# ------------------------------------------------------------
# 10. Generate Text Using Fine-Tuned Model
# ------------------------------------------------------------

# Define input prompt
prompt = "Artificial Intelligence is"

# Convert prompt into token IDs
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Disable gradient computation (inference mode)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=50,     # Maximum output length
        temperature=0.7,   # Controls randomness
        top_k=50           # Top-k sampling
    )

# Decode token IDs back to readable text
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Using device: cpu


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.17969283755580304


Step,Training Loss
50,4.625000
100,4.344500
150,4.355100
200,4.380600
250,4.301700


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Artificial Intelligence is a technology that is able to predict and control the behavior of the human brain.






























